# Phase 0: Feasibility

**Question:** can a model predict whether a detected California wildfire escalates within 24h better than a persistence rule ("it grew recently, so it keeps growing")?

Pipeline: FIRMS VIIRS detections → per-pass DBSCAN clusters → tracked fires (`fire_id`) → versioned labels → point-in-time features + Open-Meteo forecasts → baseline vs XGBoost on a time split.

All logic lives in `wildfire/`; this notebook only drives it.

In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

from wildfire import firms, fires, labels, weather, features

PROC = Path("../data/processed"); PROC.mkdir(parents=True, exist_ok=True)

## 1. Detections
Pulled once and cached as raw parquet; re-running skips anything already downloaded.

In [ ]:
firms.pull()
det = firms.load()
# why: low-confidence VIIRS pixels are mostly sun glint and hot surfaces, not fire.
# Dropping them here keeps them out of clustering, labels and features alike.
det = det[det["confidence"] != "l"]
print(len(det), "detections"); det.groupby(det.time.dt.year).size()

## 2. Fires: cluster each pass, track across passes

In [ ]:
if (PROC / "tracked.parquet").exists():
    tracked, merges = pd.read_parquet(PROC / "tracked.parquet"), pd.read_parquet(PROC / "merges.parquet")
else:
    tracked, merges = fires.track(det)
    tracked.to_parquet(PROC / "tracked.parquet"); merges.to_parquet(PROC / "merges.parquet")
summary = fires.summarize(tracked)
span = summary.groupby("fire_id")["bin"].agg(lambda b: (b.max() - b.min()).days)
print(summary.fire_id.nunique(), "fires,", len(merges), "merges,", len(summary), "fire-passes")
span.describe()

Fires that stay active for months are usually not wildfires: they're persistent industrial heat sources such as refineries, flares and power plants. They're dropped here and listed as a known issue.

In [ ]:
static = span[span > 60].index
summary = summary[~summary.fire_id.isin(static)]
print(len(static), "long-lived sources dropped")

## 3. Features and labels
Features come from `features.build`, the same function the Phase 4 scoring job will call. Labels are built for several thresholds `x`.

In [ ]:
hourly = weather.hourly_for(summary)
feat = features.build(summary, hourly)
X_SWEEP = [0.25, 0.5, 1.0]
data = {x: feat.merge(labels.label(summary, merges, x=x)[["fire_id", "bin", "escalated", "label_version"]], on=["fire_id", "bin"])
        for x in X_SWEEP}
for x, d in data.items():
    print(d.label_version.iat[0], len(d), "rows, base rate", round(d.escalated.mean(), 3))

## 4. Evaluation
- **Split by time:** train on 2022–2023, test on 2024. This mirrors deployment: predicting the future.
- **Tuning** uses `GroupKFold` by `fire_id`, so one fire never appears in both train and validation.
- **Baseline:** logistic regression on the 24h growth features only (persistence), which is calibrated by construction.
- **Metrics:** PR-AUC, because classes are imbalanced, and Brier score plus a reliability curve, because the API returns a probability.

In [ ]:
BASE = ["growth_n_24h", "growth_frp_24h"]
GRID = [dict(max_depth=d, learning_rate=lr) for d in (3, 5) for lr in (0.03, 0.1)]

def tune(tr):
    # why: grouped CV. A fire has many correlated rows; letting one fire span folds leaks.
    cv = GroupKFold(n_splits=5)
    def score(p):
        s = []
        for a, b in cv.split(tr, groups=tr.fire_id):
            m = XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.8, **p)
            m.fit(tr.iloc[a][features.FEATURES], tr.iloc[a].escalated)
            s.append(average_precision_score(tr.iloc[b].escalated, m.predict_proba(tr.iloc[b][features.FEATURES])[:, 1]))
        return np.mean(s)
    return max(GRID, key=score)

results, preds = [], {}
for x, d in data.items():
    tr, te = d[d.bin.dt.year < 2024], d[d.bin.dt.year == 2024]
    base = LogisticRegression().fit(tr[BASE], tr.escalated)
    params = tune(tr)
    model = XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.8, **params).fit(tr[features.FEATURES], tr.escalated)
    p_base, p_xgb = base.predict_proba(te[BASE])[:, 1], model.predict_proba(te[features.FEATURES])[:, 1]
    preds[x] = (te, p_base, p_xgb, model)
    results.append(dict(x=x, test_rows=len(te), base_rate=te.escalated.mean(),
                        prauc_baseline=average_precision_score(te.escalated, p_base),
                        prauc_xgb=average_precision_score(te.escalated, p_xgb),
                        brier_baseline=brier_score_loss(te.escalated, p_base),
                        brier_xgb=brier_score_loss(te.escalated, p_xgb), params=params))
results = pd.DataFrame(results); results

In [ ]:
fig, axes = plt.subplots(1, len(X_SWEEP), figsize=(5 * len(X_SWEEP), 4), sharey=True)
for ax, (x, (te, p_base, p_xgb, _)) in zip(axes, preds.items()):
    for name, p in [("baseline", p_base), ("xgboost", p_xgb)]:
        fr, mp = calibration_curve(te.escalated, p, n_bins=10, strategy="quantile")
        ax.plot(mp, fr, marker="o", label=name)
    ax.plot([0, 1], [0, 1], "k:", lw=1); ax.set(title=f"Reliability, x={x}", xlabel="predicted", ylabel="observed")
    ax.legend()
plt.tight_layout()

### Feature importance (sanity check)
Growth and weather should matter. If `month` or `age_h` dominate, the model is probably learning a data artifact instead.

In [ ]:
x_main = 0.5
imp = pd.Series(preds[x_main][3].feature_importances_, index=features.FEATURES).sort_values()
imp.plot.barh(figsize=(6, 6), title=f"XGBoost gain importance, x={x_main}");